# Case RJ - Grupo 2

**Notebook para Realização da Ingestão dos Dados para a Camada Bronze da Arquitetura Medalhão**

Nomes dos Integrantes:
- Anna Cristina
- Gabriel Guimarães
- Juan Perri
- Milton Salgado
- Richard
- Yuri

## Sobre este notebook

Ingerimos as 8 tabelas com a mesma lógica para todas — só muda o nome do arquivo e o encoding.
Fizemos isso em cima do que já estávamos rodando direto no Databricks: ajustamos o encoding certo por
arquivo (sem isso, `clientes` e `Chamados_Hora` saem com acento quebrado — são `ISO-8859-1`, não
`UTF-8`) e padronizamos o nome da tabela de destino em minúsculo.

Lemos direto da pasta do Workspace (`Shared/case-final-rocket-lab/dados`) usando o esquema `file:` no
caminho — sem isso, `spark.read` distribuído dá erro de permissão nessa pasta.

- **Fiel à origem**: tudo como `string` (`inferSchema=false`), sem renomear/corrigir nada aqui — essas
  correções ficam para a camada silver.
- **Idempotente**: escrevemos com `overwrite` — rodar de novo dá o mesmo resultado.
- **Rastreável**: toda tabela ganha `_source_file`, `_source_file_mod_ts` e `_ingestion_ts`.
- **Sem dado pessoal em output**: nenhuma célula exibe linha de `clientes` (CPF/nome/e-mail) — só
  contagem e visões agregadas.

In [ ]:
from pyspark.sql import functions as F

BASE_PATH = "/Workspace/Shared/case-final-rocket-lab/dados"
CATALOGO = "dados_vcredit"
SCHEMA_BRONZE = "default"

# arquivo -> encoding. clientes e Chamados_Hora são ISO-8859-1; os demais UTF-8.
ARQUIVOS = {
    "chamados":            "UTF-8",
    "Chamados_Hora":       "ISO-8859-1",
    "clientes":            "ISO-8859-1",
    "custos":              "UTF-8",
    "pesquisa_satisfacao": "UTF-8",
    "base_atendentes":     "UTF-8",
    "base_motivos":        "UTF-8",
    "canais":              "UTF-8",
}

## Carga e escrita

Uma função só, chamada uma vez por tabela. A tabela de destino fica sempre em minúsculo
(`<nome>_raw`), mesmo quando o nome do arquivo tem maiúscula (`Chamados_Hora` → `chamados_hora_raw`).

In [ ]:
def ingere(nome_arquivo, encoding):
    df_bronze = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .option("encoding", encoding)
        .option("quote", '"')
        .option("escape", '"')
        .load(f"file:{BASE_PATH}/{nome_arquivo}.csv")
        .select(
            "*",
            F.col("_metadata.file_path").alias("_source_file"),
            F.col("_metadata.file_modification_time").alias("_source_file_mod_ts"),
            F.current_timestamp().alias("_ingestion_ts"),
        )
    )

    destino = f"{CATALOGO}.{SCHEMA_BRONZE}.{nome_arquivo.lower()}_raw"
    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")               # idempotente: roda de novo, mesmo resultado
        .option("overwriteSchema", "true")
        .saveAsTable(destino)
    )
    print(f"{destino}: {spark.table(destino).count():,} linhas")


for nome_arquivo, encoding in ARQUIVOS.items():
    ingere(nome_arquivo, encoding)

## Verificação dos dados

Conferimos cada tabela depois da carga: contagem de linhas e uma amostra. Para `clientes` não
mostramos nenhuma linha — como ela tem CPF, nome e e-mail, exibimos só uma visão agregada por região.

Contagens esperadas (arquivos originais): chamados 134.474 · chamados_hora 131.838 · clientes 47.705 ·
custos 131.069 · pesquisa_satisfacao 81.021 · base_atendentes 25 · base_motivos 14 · canais 4.

In [ ]:
for nome_arquivo in ARQUIVOS:
    destino = f"{CATALOGO}.{SCHEMA_BRONZE}.{nome_arquivo.lower()}_raw"
    tabela = spark.table(destino)
    print(f"=== {destino}: {tabela.count():,} linhas ===")
    if nome_arquivo == "clientes":
        display(tabela.groupBy("regiao").count().orderBy("regiao"))
    else:
        display(tabela.limit(5))

## Checklist antes de aplicar uma mudança

- [ ] As contagens da verificação batem com o esperado.
- [ ] Rodamos duas vezes e o resultado foi o mesmo (idempotente).
- [ ] Nenhuma célula de verificação mostra linha de `clientes` (CPF/nome/e-mail).